In [13]:
from math import gcd
from secrets import randbits, randbelow
from typing import Tuple
import modular as mod

# ------------------------
# Utilidades aritméticas
# ------------------------
def lcm(a: int, b: int) -> int:
    return a // mod.mcd(a, b) * b

def modinv(a: int, m: int) -> int:
    # Turbo en CPython
    return mod.potencia_mod_p(a, -1, m)

# ------------------------
# Miller-Rabin (probabilístico)
# ------------------------
def _miller_rabin_witness(a: int, s: int, d: int, n: int) -> bool:
    # Devuelve True si 'a' es testigo de que n es compuesto
    x = mod.potencia_mod_p(a, d, n)
    if x == 1 or x == n - 1:
        return False
    for _ in range(s - 1):
        x = (x * x) % n
        if x == n - 1:
            return False
    return True  # compuesto

def is_probable_prime(n: int, k: int = 16) -> bool:
    if n < 2:
        return False
    # filtra pequeños
    small_primes = [2,3,5,7,11,13,17,19,23,29]
    for p in small_primes:
        if n == p:
            return True
        if n % p == 0:
            return False
    # n-1 = d * 2^s
    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1
    # k test aleatorios
    for _ in range(k):
        a = 2 + randbelow(n - 3)  # en [2, n-2]
        if _miller_rabin_witness(a, s, d, n):
            return False
    return True

def gen_prime_bits(bits: int, k: int = 16) -> int:
    assert bits >= 2
    while True:
        # número impar con bit más alto a 1
        n = (1 << (bits-1)) | randbits(bits-1) | 1
        if is_probable_prime(n, k):
            return n

def gen_prime_with_min_digits(min_digits: int, k: int = 16) -> int:
    # convierte dígitos a bits (ceil): bits ≈ dígitos * log2(10) ≈ dígitos*3.322
    bits = int(min_digits * 3.322) + 1
    return gen_prime_bits(bits, k=k)

# ------------------------
# Generación de clave RSA
# ------------------------
def rsa_keygen(min_digits_n: int = 20, k_mr: int = 16, use_e: int = 65537):
    """
    Genera (n, e, d, p, q). 'min_digits_n' controla el tamaño de n.
    """
    # Reparte el tamaño entre p y q para que n tenga al menos min_digits_n
    bits_target = int(min_digits_n * 3.322) + 1
    bits_p = bits_target // 2
    bits_q = bits_target - bits_p

    while True:
        p = gen_prime_bits(bits_p, k=k_mr)
        q = gen_prime_bits(bits_q, k=k_mr)
        if p == q:
            continue
        n = p * q
        # λ(n) = lcm(p-1, q-1)
        lam = lcm(p - 1, q - 1)
        e = use_e
        if mod.mcd(e, lam) != 1:
            # poco probable con e=65537; si ocurre, reintenta p, q
            continue
        d = modinv(e, lam)  # inversa modular rápida
        # comprueba tamaño en dígitos de n
        if len(str(n)) >= min_digits_n:
            return (n, e, d, p, q)

# ------------------------
# Cripto de juguete (sin padding)
# ------------------------
def rsa_encrypt(m: int, n: int, e: int) -> int:
    if not (0 <= m < n):
        raise ValueError("El mensaje m debe ser 0 <= m < n (usa padding/fragmentación en casos reales).")
    return mod.potencia_mod_p(m, e, n)

def rsa_decrypt(c: int, n: int, d: int) -> int:
    return mod.potencia_mod_p(c, d, n)


In [41]:
n, e, d, p, q = rsa_keygen(min_digits_n=20)  # ≈ 20+ dígitos
print("n =", n)
print("e =", e)
print("d =", d)
print("p =", p)
print("q =", q)

n = 87428730559129288649
e = 65537
d = 30759539716358238623
p = 5945376631
q = 14705330879


In [8]:
import string 
import modular as mod

n = 28282590191348679547 
e = 15780653617344828671

cList = [9319749784910798525, 1311789732010833948, 20980190400660326533, 21857556142816524701]


def ataque_texto_elegido(cList:list[int],n:int,e:int)->str:
    """Ejecuta un ataque de texto claro elegido sobre un mensaje que ha sido cifrado
    con RSA plano sin usar padding a partir de su clave pública.

    Args:
        cList (List[int]): lista de enteros que representan el mensaje cifrado
        n (int): módulo para RSA
        e (int): clave pública para RSA
    
    Returns:
        str: texto plano descifrado para el mensaje cifrado cList

    Raises:
        ValueError: Si el mensaje no se corresponde con ningún texto plano que haya sido codificado con RSA sin padding.
    """
    #idea principal. A partir de la clave publica, cremaos un codebook con todos 
    #los posibles m codificados para cada letra. Vamos rellenando el diccionario a medida
    #que recorremos la lista. 

    alfabeto = string.ascii_letters + string.digits + " "
    codebook = {}


    for i in range(256): 
        #utilizamos clave publica para generar diccionario de codigos
        chr_coded = mod.potencia_mod_p(i, e, n)
        codebook[chr_coded] = i

    print(codebook)

ataque_texto_elegido(cList, n, e)

{0: 0, 1: 1, 5082268461722105620: 2, 14080715647469805409: 3, 27580927845877702746: 4, 14066327749433081119: 5, 18056233632843985491: 6, 10481586050525438341: 7, 1337449786516161575: 8, 475200127528423438: 9, 23732211278841072713: 10, 23703140649391759507: 11, 8121081994001813535: 12, 4974864711705600697: 13, 23264300134628610039: 14, 7524874374716104556: 15, 7982861568609395518: 16, 19426663817149433009: 17, 23242182418348837170: 18, 6146074261547057623: 19, 21422858744953690968: 20, 7646380706953108642: 21, 25037150855132917749: 22, 15756300257981690960: 23, 24554183312480633739: 24, 14139232073039893691: 25, 13661252632651758592: 26, 22502324102156075146: 27, 21330661653839319778: 28, 2873394411378949125: 29, 20035434040027645529: 30, 10777172673458641945: 31, 20980190400660326533: 32, 14757100524020286400: 33, 203105126698695197: 34, 6990757063331327799: 35, 18236673516893584545: 36, 3512285221718786947: 37, 24193663596379945288: 38, 1803722035754721097: 39, 10667789970502183270: 4